In [4]:
import json, time
from pathlib import Path
import requests
import pandas as pd

RAW, SAMPLE = Path("data/raw"), Path("data/sample")
RAW.mkdir(parents=True, exist_ok=True); SAMPLE.mkdir(parents=True, exist_ok=True)
HEADERS = {"User-Agent": "Max Hostetter; maho3360@colorado.edu"}

# COLLECT: one request per year, 1-second pause, raw JSON saved to data/raw/ (skipped if already saved)
for year in range(2019, 2027):
    path = RAW / f"federal_register_{year}.json"
    if path.exists():
        continue
    r = requests.get("https://www.federalregister.gov/api/v1/documents.json", headers=HEADERS, timeout=60, 
        params={
        "conditions[term]": '"artificial intelligence"', "per_page": 1000,
        "conditions[publication_date][gte]": f"{year}-01-01", "conditions[publication_date][lte]": f"{year}-12-31",
        "fields[]": ["document_number", "title", "type", "agency_names", "publication_date"]})
    r.raise_for_status()
    path.write_text(r.text, encoding="utf-8")
    time.sleep(1)

# CLEAN: one row per document
raw = pd.DataFrame([d for p in sorted(RAW.glob("federal_register_*.json"))
                    for d in json.loads(p.read_text(encoding="utf-8"))["results"]])
fed = raw.drop_duplicates("document_number").copy()
fed["agency_names"] = fed.agency_names.apply(lambda a: "; ".join(filter(None, a or [])))
fed["publication_date"] = pd.to_datetime(fed.publication_date)
fed["about_ai"] = fed.title.str.contains(r"artificial intelligence|\bAI\b", case=False)
fed = fed.sort_values("publication_date")
fed.to_csv("data/federal_register_ai.csv", index=False)
fed.head()

,document_number,title,type,agency_names,publication_date,about_ai
20,2019-02204,Public Forum,Notice,Privacy and Civil Liberties Oversight Board,2019-02-13,False
2,2019-02544,Maintaining American Leadership in Artificial ...,Presidential Document,Executive Office of the President,2019-02-14,True
30,2019-02546,Defense Innovation Board; Notice of Federal Ad...,Notice,Defense Department; Office of the Secretary,2019-02-15,False
48,2019-02224,"21st Century Cures Act: Interoperability, Info...",Proposed Rule,Health and Human Services Department; Office o...,2019-03-04,False
36,2019-05116,Notice of Third Public Meeting on Developing t...,Notice,Commerce Department; Patent and Trademark Offi...,2019-03-19,False
